# Step 4: Direction Analysis — Steering vs Inter-Persona

Investigate the geometric relationship between steering directions and the
inter-persona axis in activation space.

Questions:
- Are steering vectors aligned with, orthogonal to, or independent of the persona axis?
- Does the alignment vary by trait?
- What does this tell us about how traits and personas are represented?

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from persona_steering.config import VECTORS_DIR, Trait
from persona_steering.personas import load_all_personas, PersonaInducer
from persona_steering.analysis import compare_steering_vs_interpersona, decompose_shared_specific
from persona_steering.utils import load_pickle, ensure_output_dirs

ensure_output_dirs()
all_vectors = load_pickle(VECTORS_DIR / "all_vectors.pkl")
personas = load_all_personas()
persona_slugs = [p.slug for p in personas]

In [ ]:
# Compute inter-persona axes from cached activations
# (Requires running notebook 01 first to have activations cached,
# or loading from saved activations)

# For now, approximate the inter-persona direction as the difference
# between mean steering vectors across all traits
sample_trait = list(all_vectors[persona_slugs[0]].keys())[0]
sample_layers = sorted(all_vectors[persona_slugs[0]][sample_trait].keys())
mid_layer = sample_layers[len(sample_layers) // 2]

# Compute mean vector per persona (across traits)
persona_means = {}
for slug in persona_slugs:
    vecs = [all_vectors[slug][t][mid_layer].vector for t in Trait
            if t in all_vectors[slug] and mid_layer in all_vectors[slug][t]]
    if vecs:
        persona_means[slug] = torch.stack(vecs).mean(dim=0)

# Inter-persona axes
persona_axes = {}
for i, pa in enumerate(persona_slugs):
    for j, pb in enumerate(persona_slugs):
        if j <= i:
            continue
        if pa in persona_means and pb in persona_means:
            persona_axes[(pa, pb)] = persona_means[pb] - persona_means[pa]
            print(f"Persona axis {pa} -> {pb}: magnitude={persona_axes[(pa, pb)].norm():.4f}")

In [ ]:
# Compare each trait's steering direction against persona axes
results = []

for trait in Trait:
    for slug in persona_slugs:
        vec = all_vectors.get(slug, {}).get(trait, {}).get(mid_layer)
        if vec is None:
            continue
        for (pa, pb), axis in persona_axes.items():
            comp = compare_steering_vs_interpersona(vec, axis)
            comp["trait"] = trait.value
            comp["persona"] = slug
            comp["axis"] = f"{pa}->{pb}"
            results.append(comp)

import pandas as pd
df = pd.DataFrame(results)
print(df.groupby(["trait", "axis"])["cosine_similarity"].mean())

In [ ]:
# Visualise alignment by trait
fig, ax = plt.subplots(figsize=(10, 6))

pivot = df.pivot_table(values="cosine_similarity", index="trait",
                       columns="persona", aggfunc="mean")
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="RdBu_r", center=0, ax=ax)
ax.set_title("Steering-Persona Axis Alignment by Trait")
plt.tight_layout()
plt.savefig("../outputs/figures/steering_persona_alignment.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Alignment ratio: what fraction of steering magnitude lies along persona axis?
fig, ax = plt.subplots(figsize=(8, 5))

pivot_ratio = df.pivot_table(values="alignment_ratio", index="trait",
                             columns="persona", aggfunc="mean")
pivot_ratio.plot(kind="bar", ax=ax)
ax.set_ylabel("Alignment Ratio (|proj| / |steering|)")
ax.set_title("Fraction of Steering Vector Aligned with Persona Axis")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.savefig("../outputs/figures/alignment_ratios.png", dpi=150, bbox_inches="tight")
plt.show()